In [2]:
#Refer: 0.1.2-Langchain_simple_exmpl.ipynb for setup.
#!pip install torch==2.2.2
#!pip install transformers==4.37.2
#if any issues with older installation, uninstall and reinstall approprite version
#!pip uninstall transformers torch huggingface-hub tokenizers -y
#!pip uninstall transformers -y
#!pip install torch==2.2.2
#!pip install transformers==4.37.2

In [3]:
import transformers
print(transformers.__version__)
print(transformers.__file__)
import os
import transformers.models.t5
print(transformers.models.t5.__file__)

4.57.1
C:\Users\Ajay\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\__init__.py
C:\Users\Ajay\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\models\t5\__init__.py


In [4]:
from transformers import pipeline
pipe = pipeline("text2text-generation", model="google/flan-t5-large")


Device set to use cpu


In [5]:
prompt = "Write a catchy tagline for a coffee brand."
print(pipe(prompt)[0]["generated_text"])

a cup of coffee a day keeps the doctor away


In [7]:
brand_name = input("Enter the brand name: ")
product_name = input("Enter the product name: ")
product_category = input("Enter the product category (e.g., drink, oil, cream, bar): ")
product_description = input("Enter a short description of the product: ")
key_features = input("List the key product features (comma-separated): ")
revenue_target = input("Enter the target revenue increase (e.g., '24%'): ")
target_age_range = input("Enter the target age range (e.g., '22–55 years'): ")
target_interests = input("Enter target audience interests (comma-separated): ")
target_pain_points = input("Enter key audience pain points: ")
campaign_focus = input("Enter the campaign focus (e.g., 'Brand Awareness', 'Pre-orders', 'Seasonal Launch'): ")



Enter the brand name:  Myprot
Enter the product name:  health bar
Enter the product category (e.g., drink, oil, cream, bar):  bar
Enter a short description of the product:  Something that keeps you full while keeping u empty
List the key product features (comma-separated):  crunchy, healthy, low-cab, high-energy
Enter the target revenue increase (e.g., '24%'):  24%
Enter the target age range (e.g., '22–55 years'):  30-45
Enter target audience interests (comma-separated):  weight-concious, health-concious, 
Enter key audience pain points:  weight loss without muscle loss
Enter the campaign focus (e.g., 'Brand Awareness', 'Pre-orders', 'Seasonal Launch'):  Brand Awareness


In [8]:
# Construct the dynamic prompt
prompt = f"""
You are a senior marketing strategist with 25+ years of experience in the wellness, fitness, and healthy lifestyle industry.

Create a comprehensive product launch campaign for a new product.

**Context:**
{brand_name} aims to strengthen its position in the wellness and fitness market and achieve at least a {revenue_target} revenue increase compared to the previous year.
The campaign should focus on brand differentiation, audience engagement, and conversion.

**Product Details:**
- Product Name: {product_name}
- Product Category: {product_category}
- Description: {product_description}
- Key Features: {key_features}

**Target Audience:**
- Age Range: {target_age_range}
- Interests: {target_interests}
- Pain Points: {target_pain_points}

**Competitor Context:**
Analyze 3 leading competitors in the same product category from major platforms (e.g., Amazon, health-food retailers, or direct-to-consumer brands).
Highlight:
- Strengths
- Weaknesses
- Opportunities for {brand_name} to differentiate.

**Campaign Focus:** {campaign_focus}

**Deliverables:**
1. Campaign strategy summary (positioning, message, and insights)
2. Tagline
3. Instagram post (visual concept + caption)
4. Call-to-action (for pre-orders or early adoption, including any limited-time offer)
5. Differentiation strategy and growth recommendations to meet the revenue goal.

**Tone & Style:** Confident, aspirational, and aligned with modern wellness branding — a balance of science-backed credibility and lifestyle inspiration.
"""

In [9]:
prompt

'\nYou are a senior marketing strategist with 25+ years of experience in the wellness, fitness, and healthy lifestyle industry.\n\nCreate a comprehensive product launch campaign for a new product.\n\n**Context:**\nMyprot aims to strengthen its position in the wellness and fitness market and achieve at least a 24% revenue increase compared to the previous year.\nThe campaign should focus on brand differentiation, audience engagement, and conversion.\n\n**Product Details:**\n- Product Name: health bar\n- Product Category: bar\n- Description: Something that keeps you full while keeping u empty\n- Key Features: crunchy, healthy, low-cab, high-energy\n\n**Target Audience:**\n- Age Range: 30-45\n- Interests: weight-concious, health-concious, \n- Pain Points: weight loss without muscle loss\n\n**Competitor Context:**\nAnalyze 3 leading competitors in the same product category from major platforms (e.g., Amazon, health-food retailers, or direct-to-consumer brands).\nHighlight:\n- Strengths\n- 

In [ ]:
print(pipe(prompt)[0]["generated_text"])

In [11]:
#Switching to GPT models and Azure/OpenAI
#OpenAI SDK (openai.AzureOpenAI)
import os
import dotenv
import openai
from openai import AzureOpenAI
from dotenv import load_dotenv

#load_dotenv("/content/.env")
load_dotenv()

# Initialize client once
client = AzureOpenAI(
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

deployment_name = os.getenv("AZURE_DEPLOYMENT_NAME")

In [12]:
def get_completion(prompt, deployment_name=deployment_name):
    """
    Get a chat completion from Azure OpenAI.
    Args:
        prompt (str): User input prompt.
        deployment_name (str): The deployment name you gave your model in Azure portal.
    Returns:
        dict: Full response object, or error dict.
    """
    try:
        messages = [{"role": "user", "content": prompt}]
        response = client.chat.completions.create(
            model=deployment_name,    # <-- This is the "deployment name" not the raw model name
            messages=messages,
            temperature=0.1,
            top_p=0.8,
            max_tokens=512
        )

        return response.model_dump()  # Return the full response as dict

    except Exception as e:
        return {"error": str(e)}

In [ ]:
get_completion(prompt)

In [14]:
#Or
response = get_completion(prompt)

In [ ]:
print(response['choices'][0]['message']['content'])

In [16]:
#If using langchain & AzureOpenAI
from langchain_openai import AzureOpenAI
# Initialize client once
client_lc = AzureOpenAI(
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name="gpt-4.1", # chat completions would work with this model
    temperature=0.5,
    top_p=0.8,
    max_tokens=512
)

In [21]:
def get_completion_lang_azure(prompt):
    try:
        response = client_lc.invoke(prompt)
        return response
    except Exception as e:
        return {"error": str(e)}

In [22]:
response_lc = get_completion_lang_azure(prompt)

In [23]:
print(response_lc)

{'error': "Error code: 400 - {'error': {'code': 'OperationNotSupported', 'message': 'The completion operation does not work with the specified model, gpt-4.1. Please choose different model and try again. You can learn more about which models can be used with each operation here: https://go.microsoft.com/fwlink/?linkid=2197993.'}}"}


In [ ]:
#We can use different newer model to test with AzureOpenAI from langchain or proceed with AzureChatOpenAI

In [24]:
#Use LangChain Chat model , If you want messages-style chat
from langchain_openai import AzureChatOpenAI

client_lc = AzureChatOpenAI(
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name="gpt-4.1",
    temperature=0.5,
    max_tokens=512
)

In [25]:
def get_completion_lang_azure(prompt):
    try:
        response = client_lc.invoke(prompt)
        return response.content
    except Exception as e:
        return {"error": str(e)}

In [26]:
response_lc = get_completion_lang_azure(prompt)

In [ ]:
print(response_lc)